# Dataset 4: Thursday-Morning-WebAttacks EDA
This notebook inspects Web Attacks (Brute Force, XSS, and SQL Injection) and addresses label encoding artifacts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

# Label sanitization for Windows-1252 byte 0x96
mapping = {
    'Web Attack  Brute Force': 'Web Attack - Brute Force',
    'Web Attack  XSS': 'Web Attack - XSS',
    'Web Attack  Sql Injection': 'Web Attack - Sql Injection'
}
df['Normalized_Label'] = df['Label'].replace(mapping).str.strip()

print(f"Loaded Thursday Web Attacks: {df.shape[0]:,} rows")
print(f"Normalized Class breakdown:\n{df['Normalized_Label'].value_counts()}")


## 1. Web Attack Payload Sizes (Request vs Response)

In [ ]:
for c in [c for c in df.columns if c not in ['Label', 'Normalized_Label']]:
    df[c] = pd.to_numeric(df[c], errors='coerce')

plt.figure(figsize=(9, 4.5))
df['log_Fwd_Length'] = np.log10(df['Total Length of Fwd Packets'].clip(lower=1))
sns.boxplot(data=df, x='Normalized_Label', y='log_Fwd_Length', palette='Set2')
plt.xticks(rotation=15, ha='right')
plt.title("Thursday Morning: Total Fwd Packet Length (Payload Size) by Class")
plt.show()


## 2. Key Findings & Preprocessing Recommendations
- Label parser must sanitize non-ASCII en-dash `` bytes.
- SQL Injection (21 flows) requires grouping under 'Web Attack' meta-class or SMOTE.
- All web attacks targeted Port 80 HTTP web server.
